Here I will try to make llm to plan and solve a problem.

In [15]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [16]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        pass



In [17]:
openai = OpenAI()

In [18]:
# create some lists of todos and completed
todos = []
completed = []

In [19]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result


In [20]:
get_todo_report()

''

In [23]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [24]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [25]:
todos, completed = [], []
create_todos(["Buy water", "Go for a walk with Klyapi", "Eat qattama"])


Todo #1: Buy water
Todo #2: Go for a walk with Klyapi
Todo #3: Eat qattama

'Todo #1: Buy water\nTodo #2: Go for a walk with Klyapi\nTodo #3: Eat qattama\n'

In [26]:
mark_complete(1, "I stacked water in the fridge")

I stacked water in the fridge

Todo #1: Buy water
Todo #2: Go for a walk with Klyapi
Todo #3: Eat qattama

'Todo #1: [green][strike]Buy water[/strike][/green]\nTodo #2: Go for a walk with Klyapi\nTodo #3: Eat qattama\n'

In [27]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [28]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [29]:
tools = [{"type": "function", "function": create_todos_json},
         {"type": "function", "function": mark_complete_json}]

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [38]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)
    

In [39]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
Write a method in java, islandCount, that takes in a grid containing Ws and Ls. W represents water and L represents land. The method should return the number of islands on the grid.
"""
messages = [{"role": "system", "content": system_message}, {"role":"user", "content":user_message}]

In [40]:
todos, completed = [],[]
loop(messages)

Todo #1: Outline algorithm (DFS/BFS flood fill) and edge cases for counting islands in a char grid of 'W'/'L'.
Todo #2: Write Java method signature and implementation details (visited tracking, bounds checks, iteration).
Todo #3: Provide final solution formatted in Rich console markup (no code blocks).

Algorithm
- Traverse every cell.
- When an unvisited 'L' is found, increment island count and run flood fill (DFS/BFS) to mark all connected land 
(4-directionally: up/down/left/right) as visited.
- Water cells ('W') are ignored.

Edge cases
- Empty grid or null → return 0.
- Non-rectangular (jagged) arrays: use grid.length per row.
- Avoid stack overflow on huge grids by using iterative BFS/stack if desired; recursive DFS is fine for typical 
interview-sized grids.

Todo #1: Outline algorithm (DFS/BFS flood fill) and edge cases for counting islands in a char grid of 'W'/'L'.
Todo #2: Write Java method signature and implementation details (visited tracking, bounds checks, iteration).
Todo #3: Provide final solution formatted in Rich console markup (no code blocks).

Implementation plan
- Method: int islandCount(char[][] grid)
- Maintain boolean[][] visited sized to grid.length x grid.length per row.
- Nested loop over rows/cols:
  - If grid == 'L' and not visited → count++, call dfs(r,c).
- dfs:
  - Return if out of bounds, cell is 'W', or visited.
  - Mark visited true, recurse to 4 neighbors.
- Bounds: r < 0 || r >= grid.length || c < 0 || c >= grid.length.

Todo #1: Outline algorithm (DFS/BFS flood fill) and edge cases for counting islands in a char grid of 'W'/'L'.
Todo #2: Write Java method signature and implementation details (visited tracking, bounds checks, iteration).
Todo #3: Provide final solution formatted in Rich console markup (no code blocks).

Final response includes Java method + helper, formatted without code fences.

Todo #1: Outline algorithm (DFS/BFS flood fill) and edge cases for counting islands in a char grid of 'W'/'L'.
Todo #2: Write Java method signature and implementation details (visited tracking, bounds checks, iteration).
Todo #3: Provide final solution formatted in Rich console markup (no code blocks).

Java: count islands with DFS flood fill (4-directional)

public static int islandCount(char[][] grid) {
    if (grid == null || grid.length == 0) return 0;

    boolean[][] visited = new boolean[];
    for (int r = 0; r < grid.length; r++) {
        visited = new boolean[grid.length]; // supports jagged rows too
    }

    int count = 0;

    for (int r = 0; r < grid.length; r++) {
        for (int c = 0; c < grid.length; c++) {
            if (grid == 'L' && !visited) {
                count++;
                dfs(grid, visited, r, c);
            }
        }
    }
    return count;
}

private static void dfs(char[][] grid, boolean[][] visited, int r, int c) {
    if (r < 0 || r >= grid.length) return;
    if (c < 0 || c >= grid.length) return;
    if (visited) return;
    if (grid == 'W') return;

    visited = true;

    dfs(grid, visited, r - 1, c); // up
    dfs(grid, visited, r + 1, c); // down
    dfs(grid, visited, r, c - 1); // left
    dfs(grid, visited, r, c + 1); // right
}

Result: returns the number of connected components of 'L' cells (islands), where connectivity is 
up/down/left/right.